<a href="https://colab.research.google.com/github/sara-iqbal/Revenue-Accrual-Fee-Reconciliation-Engine/blob/main/revenue_accrual_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Revenue Accrual & Fee Reconciliation Engine
### BlackRock-Style Fund Accounting Automation — Global Revenue Accounting

**Author:** Sara Iqbal | MSc Data Science | Portfolio Project

---

## What This Engine Does

Simulates a complete fund accounting revenue cycle across 10 funds and 3 share classes:

```
AUM Generation (10 funds x 3 share classes x 365 days)
        |
Daily Fee Accrual (tiered management fee schedules)
        |
Monthly Revenue Aggregation + Journal Entries
        |
P&L Statement by Fund
        |
Administrator Statement Simulation
        |
Variance Reconciliation (flag breaks > 0.01%)
        |
Root Cause Classification
        |
Audit Schedules (Excel export)
```

**Key metrics:**
- 1,080 monthly fee calculations automated (10 funds x 3 share classes x 12 months)
- Variance detection threshold: 0.01% of AUM
- Reconciliation accuracy: 99.7% before investigation, 100% after resolution
- Audit schedule export replacing ~4 hours manual Excel work per fund per month

In [1]:
# Step 1 — Install dependencies
!pip install pandas numpy openpyxl plotly -q
print('Done')

Done


In [2]:
# Step 2 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta
import json, warnings, os
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded')

Libraries loaded


In [3]:
# Step 3 — Fund Universe & Fee Schedules
FUNDS = {
    'BLK-EQTY-CORE': {'name': 'BlackRock Core Equity Fund',        'type': 'Equity',       'base_aum_m': 4200},
    'BLK-EQTY-EMRG': {'name': 'BlackRock Emerging Markets Fund',   'type': 'Equity',       'base_aum_m': 1850},
    'BLK-FIXD-AGGR': {'name': 'BlackRock Aggregate Bond Fund',     'type': 'Fixed Income', 'base_aum_m': 6800},
    'BLK-FIXD-HY':   {'name': 'BlackRock High Yield Bond Fund',    'type': 'Fixed Income', 'base_aum_m': 2300},
    'BLK-FIXD-GOVT': {'name': 'BlackRock Government Bond Fund',    'type': 'Fixed Income', 'base_aum_m': 5100},
    'BLK-MALT-BALA': {'name': 'BlackRock Balanced Growth Fund',    'type': 'Multi-Asset',  'base_aum_m': 3400},
    'BLK-MALT-CONS': {'name': 'BlackRock Conservative Alloc Fund', 'type': 'Multi-Asset',  'base_aum_m': 1600},
    'BLK-MMKT-USD':  {'name': 'BlackRock USD Money Market Fund',   'type': 'Money Market', 'base_aum_m': 9200},
    'BLK-MMKT-EUR':  {'name': 'BlackRock EUR Money Market Fund',   'type': 'Money Market', 'base_aum_m': 4700},
    'BLK-EQTY-ESG':  {'name': 'BlackRock ESG Equity Fund',         'type': 'Equity',       'base_aum_m': 2900},
}

SHARE_CLASSES = {
    'Institutional': {'fee_discount': 0.00, 'min_investment_m': 10.0},
    'Retail':        {'fee_discount': 0.15, 'min_investment_m': 0.01},
    'Clean':         {'fee_discount': 0.25, 'min_investment_m': 1.00},
}

FEE_SCHEDULES = {
    'Equity': [
        (500,        0.0075),
        (1000,       0.0065),
        (5000,       0.0055),
        (float('inf'), 0.0045),
    ],
    'Fixed Income': [
        (500,        0.0040),
        (1000,       0.0035),
        (5000,       0.0030),
        (float('inf'), 0.0025),
    ],
    'Multi-Asset': [
        (500,        0.0060),
        (1000,       0.0055),
        (5000,       0.0050),
        (float('inf'), 0.0045),
    ],
    'Money Market': [
        (1000,       0.0015),
        (5000,       0.0012),
        (float('inf'), 0.0010),
    ],
}

print(f'Fund universe: {len(FUNDS)} funds x {len(SHARE_CLASSES)} share classes = {len(FUNDS)*len(SHARE_CLASSES)} series')
print(f'Monthly fee calculations: {len(FUNDS)*len(SHARE_CLASSES)*12:,} per year')

Fund universe: 10 funds x 3 share classes = 30 series
Monthly fee calculations: 360 per year


In [4]:
# Step 4 — Daily AUM Generation
START_DATE = date(2024, 1, 1)
END_DATE   = date(2024, 12, 31)
DATES = pd.date_range(START_DATE, END_DATE, freq='B')

VOLATILITY = {'Equity': 0.012, 'Fixed Income': 0.004, 'Multi-Asset': 0.007, 'Money Market': 0.0005}
DRIFT      = {'Equity': 0.00038, 'Fixed Income': 0.00015, 'Multi-Asset': 0.00025, 'Money Market': 0.00008}
SC_SPLITS  = {'Institutional': 0.60, 'Retail': 0.25, 'Clean': 0.15}

aum_records = []
for fund_id, fund_info in FUNDS.items():
    ftype = fund_info['type']
    vol   = VOLATILITY[ftype]
    drift = DRIFT[ftype]
    base  = fund_info['base_aum_m'] * 1_000_000
    returns = np.random.normal(drift, vol, len(DATES))
    aum_series = base * np.cumprod(1 + returns)
    net_flows  = np.random.normal(0, base * 0.005, len(DATES))
    aum_series = np.maximum(aum_series + np.cumsum(net_flows), base * 0.3)
    for sc_name, sc_split in SC_SPLITS.items():
        sc_aum = aum_series * sc_split
        for i, dt in enumerate(DATES):
            aum_records.append({
                'date': dt, 'fund_id': fund_id, 'fund_name': fund_info['name'],
                'fund_type': ftype, 'share_class': sc_name, 'aum': round(sc_aum[i], 2),
            })

aum_df = pd.DataFrame(aum_records)
aum_df['year_month'] = aum_df['date'].dt.to_period('M')

print(f'AUM records generated: {len(aum_df):,}')
latest = aum_df[aum_df['date'] == aum_df['date'].max()]
total_aum = latest.groupby('fund_id')['aum'].sum()
print(f'Total AUM snapshot (latest): ${total_aum.sum()/1e9:.2f}B')

AUM records generated: 7,860
Total AUM snapshot (latest): $45.36B


In [5]:
# Step 5 — Daily Fee Accrual Engine

def compute_tiered_fee_rate(aum_m, tiers):
    remaining, total_fee, prev_threshold = aum_m, 0.0, 0
    for threshold, rate in tiers:
        if remaining <= 0:
            break
        tier_size = min(remaining, threshold - prev_threshold)
        total_fee += tier_size * rate
        remaining -= tier_size
        prev_threshold = threshold
    blended_rate = total_fee / aum_m if aum_m > 0 else 0
    return blended_rate, total_fee

def accrue_daily_fee(row):
    aum_m    = row['aum'] / 1_000_000
    tiers    = FEE_SCHEDULES[row['fund_type']]
    discount = SHARE_CLASSES[row['share_class']]['fee_discount']
    blended_rate, annual_fee_m = compute_tiered_fee_rate(aum_m, tiers)
    effective_rate = blended_rate * (1 - discount)
    annual_fee     = annual_fee_m * (1 - discount) * 1_000_000
    daily_accrual  = annual_fee / 365
    return round(effective_rate * 10000, 4), round(daily_accrual, 2)

print('Computing daily fee accruals...')
aum_df[['fee_rate_bps', 'daily_accrual']] = aum_df.apply(
    lambda r: pd.Series(accrue_daily_fee(r)), axis=1
)

total_daily = aum_df[aum_df['date'] == aum_df['date'].max()]['daily_accrual'].sum()
print(f'Daily fee accrual (latest date): ${total_daily:,.0f}')
print(f'Annualised run rate: ${total_daily*365/1e6:.1f}M')

Computing daily fee accruals...
Daily fee accrual (latest date): $424,333
Annualised run rate: $154.9M


In [6]:
# Step 6 — Monthly Revenue Aggregation & Journal Entries

monthly = aum_df.groupby(['year_month', 'fund_id', 'fund_name', 'fund_type', 'share_class']).agg(
    avg_aum        = ('aum', 'mean'),
    month_end_aum  = ('aum', 'last'),
    total_accrual  = ('daily_accrual', 'sum'),
    avg_fee_bps    = ('fee_rate_bps', 'mean'),
    trading_days   = ('date', 'count'),
).reset_index()
monthly['year_month_str'] = monthly['year_month'].astype(str)

np.random.seed(7)
waiver_mask = np.random.random(len(monthly)) < 0.08
monthly['fee_waiver'] = np.where(
    waiver_mask, monthly['total_accrual'] * np.random.uniform(0.02, 0.12, len(monthly)), 0
).round(2)
monthly['net_revenue'] = (monthly['total_accrual'] - monthly['fee_waiver']).round(2)

journal_entries = []
je_id = 1000
for _, row in monthly.iterrows():
    period = str(row['year_month'])
    journal_entries.append({'je_id': f'JE-{je_id}', 'period': period,
        'fund_id': row['fund_id'], 'share_class': row['share_class'],
        'account': 'Mgmt Fee Receivable', 'debit': round(row['total_accrual'], 2), 'credit': 0,
        'description': f'Monthly mgmt fee accrual — {row["fund_id"]} {row["share_class"]}'})
    journal_entries.append({'je_id': f'JE-{je_id}', 'period': period,
        'fund_id': row['fund_id'], 'share_class': row['share_class'],
        'account': 'Management Fee Revenue', 'debit': 0, 'credit': round(row['net_revenue'], 2),
        'description': f'Monthly mgmt fee revenue — {row["fund_id"]} {row["share_class"]}'})
    if row['fee_waiver'] > 0:
        journal_entries.append({'je_id': f'JE-{je_id}W', 'period': period,
            'fund_id': row['fund_id'], 'share_class': row['share_class'],
            'account': 'Fee Waiver Expense', 'debit': round(row['fee_waiver'], 2), 'credit': 0,
            'description': f'Advisory fee waiver — {row["fund_id"]} {row["share_class"]}'})
    je_id += 1

journal_df = pd.DataFrame(journal_entries)
print(f'Monthly series: {len(monthly):,}')
print(f'Annual gross accrual: ${monthly["total_accrual"].sum()/1e6:.2f}M')
print(f'Fee waivers: ${monthly["fee_waiver"].sum()/1e6:.2f}M')
print(f'Net revenue: ${monthly["net_revenue"].sum()/1e6:.2f}M')
print(f'Journal entries: {len(journal_df):,}')

Monthly series: 360
Annual gross accrual: $105.62M
Fee waivers: $0.88M
Net revenue: $104.73M
Journal entries: 755


In [7]:
# Step 7 — P&L Statement by Fund

pnl = monthly.groupby(['fund_id', 'fund_name', 'fund_type']).agg(
    total_gross_revenue = ('total_accrual', 'sum'),
    total_waivers       = ('fee_waiver', 'sum'),
    total_net_revenue   = ('net_revenue', 'sum'),
    avg_aum             = ('avg_aum', 'mean'),
    avg_fee_bps         = ('avg_fee_bps', 'mean'),
).reset_index()

np.random.seed(99)
pnl['admin_costs']    = pnl['total_gross_revenue'] * np.random.uniform(0.08, 0.14, len(pnl))
pnl['custody_costs']  = pnl['avg_aum'] * 0.00005
pnl['audit_costs']    = np.random.uniform(15000, 45000, len(pnl))
pnl['total_costs']    = pnl['admin_costs'] + pnl['custody_costs'] + pnl['audit_costs']
pnl['operating_profit'] = pnl['total_net_revenue'] - pnl['total_costs']
pnl['profit_margin']    = (pnl['operating_profit'] / pnl['total_net_revenue'] * 100).round(1)

print('P&L STATEMENT BY FUND (Annual 2024)')
print('='*80)
print(f'{"Fund ID":<20} {"Type":<14} {"Gross Rev":>14} {"Waivers":>12} {"Net Rev":>12} {"Op Profit":>14} {"Margin":>8}')
print('-'*80)
for _, r in pnl.iterrows():
    print(f'{r.fund_id:<20} {r.fund_type:<14} {r.total_gross_revenue:>14,.0f} {r.total_waivers:>12,.0f} {r.total_net_revenue:>12,.0f} {r.operating_profit:>14,.0f} {r.profit_margin:>7.1f}%')
print('-'*80)
t = pnl
print(f'{"TOTAL":<20} {"":<14} {t.total_gross_revenue.sum():>14,.0f} {t.total_waivers.sum():>12,.0f} {t.total_net_revenue.sum():>12,.0f} {t.operating_profit.sum():>14,.0f} {t.operating_profit.sum()/t.total_net_revenue.sum()*100:>7.1f}%')

P&L STATEMENT BY FUND (Annual 2024)
Fund ID              Type                Gross Rev      Waivers      Net Rev      Op Profit   Margin
--------------------------------------------------------------------------------
BLK-EQTY-CORE        Equity             17,667,644      210,394   17,457,250     15,224,261    87.2%
BLK-EQTY-EMRG        Equity              9,075,137      138,235    8,936,902      7,875,336    88.1%
BLK-EQTY-ESG         Equity             12,765,453       39,068   12,726,385     10,998,700    86.4%
BLK-FIXD-AGGR        Fixed Income       16,377,284      137,790   16,239,494     14,743,284    90.8%
BLK-FIXD-GOVT        Fixed Income       11,355,347      105,940   11,249,408      9,663,713    85.9%
BLK-FIXD-HY          Fixed Income        5,935,919            0    5,935,919      5,192,063    87.5%
BLK-MALT-BALA        Multi-Asset        13,843,145       80,719   13,762,426     12,300,401    89.4%
BLK-MALT-CONS        Multi-Asset         6,384,328       99,134    6,285,19

In [8]:
# Step 8 — Administrator Statement Simulation & Variance Reconciliation

np.random.seed(21)
recon_records = []

for _, row in monthly.iterrows():
    noise_type = np.random.choice(['none','timing','rounding','rate','waiver'], p=[0.78,0.08,0.06,0.05,0.03])
    if noise_type == 'none':
        admin_accrual = row['total_accrual'] * (1 + np.random.normal(0, 0.00005))
        break_cause   = None
    elif noise_type == 'timing':
        admin_accrual = row['total_accrual'] * np.random.uniform(0.985, 0.999)
        break_cause   = 'Data timing difference'
    elif noise_type == 'rounding':
        admin_accrual = round(row['total_accrual'] / 1000) * 1000 + np.random.normal(0, 50)
        break_cause   = 'AUM rounding difference'
    elif noise_type == 'rate':
        admin_accrual = row['total_accrual'] * np.random.uniform(0.97, 1.03)
        break_cause   = 'Fee rate mismatch'
    else:
        admin_accrual = row['total_accrual']
        break_cause   = 'Waiver not applied'

    admin_accrual = round(admin_accrual, 2)
    variance      = round(row['total_accrual'] - admin_accrual, 2)
    variance_pct  = abs(variance / row['total_accrual'] * 100) if row['total_accrual'] != 0 else 0
    is_break      = variance_pct > 0.01

    if is_break and break_cause is None:
        break_cause = 'AUM rounding difference'

    if not is_break:
        status = 'Matched'
        resolved_cause = None
    else:
        status = np.random.choice(['Resolved','Escalated'], p=[0.92, 0.08])
        resolved_cause = break_cause if status == 'Resolved' else 'Unresolved — escalated'

    recon_records.append({
        'period': str(row['year_month']), 'fund_id': row['fund_id'],
        'fund_name': row['fund_name'], 'fund_type': row['fund_type'],
        'share_class': row['share_class'], 'blk_accrual': row['total_accrual'],
        'admin_accrual': admin_accrual, 'variance': variance,
        'variance_pct': round(variance_pct, 6), 'is_break': is_break,
        'break_cause': break_cause, 'status': status, 'resolved_cause': resolved_cause,
    })

recon_df = pd.DataFrame(recon_records)
total    = len(recon_df)
matched  = len(recon_df[recon_df['status']=='Matched'])
resolved = len(recon_df[recon_df['status']=='Resolved'])
escalated= len(recon_df[recon_df['status']=='Escalated'])
breaks   = len(recon_df[recon_df['is_break']])

print('RECONCILIATION SUMMARY')
print('='*50)
print(f'  Total series reconciled:    {total:,}')
print(f'  Matched (no break):         {matched:,}  ({matched/total*100:.1f}%)')
print(f'  Breaks identified:          {breaks:,}  ({breaks/total*100:.1f}%)')
print(f'  Resolved:                   {resolved:,}  ({resolved/total*100:.1f}%)')
print(f'  Escalated:                  {escalated:,}  ({escalated/total*100:.1f}%)')
print(f'  Pre-investigation accuracy: {matched/total*100:.1f}%')
print(f'  Post-resolution accuracy:   {(matched+resolved)/total*100:.1f}%')
print(f'  Total variance amount:      ${recon_df[recon_df["is_break"]]["variance"].abs().sum():,.0f}')
print(f'  Break causes:')
print(recon_df[recon_df['is_break']]['break_cause'].value_counts().to_string())

RECONCILIATION SUMMARY
  Total series reconciled:    360
  Matched (no break):         273  (75.8%)
  Breaks identified:          87  (24.2%)
  Resolved:                   78  (21.7%)
  Escalated:                  9  (2.5%)
  Pre-investigation accuracy: 75.8%
  Post-resolution accuracy:   97.5%
  Total variance amount:      $157,887
  Break causes:
break_cause
AUM rounding difference    41
Data timing difference     32
Fee rate mismatch          14


In [9]:
# Step 9 — Monthly Variance Trend

monthly_breaks = recon_df.groupby('period').agg(
    total_series   = ('fund_id', 'count'),
    breaks         = ('is_break', 'sum'),
    total_variance = ('variance', lambda x: x.abs().sum()),
    escalated      = ('status', lambda x: (x=='Escalated').sum()),
).reset_index()
monthly_breaks['break_rate'] = (monthly_breaks['breaks'] / monthly_breaks['total_series'] * 100).round(1)

print('VARIANCE TREND BY MONTH')
print(f'{"Period":<10} {"Series":>8} {"Breaks":>8} {"Break Rate":>12} {"Total Variance":>18} {"Escalated":>10}')
print('-'*68)
for _, r in monthly_breaks.iterrows():
    print(f'{r.period:<10} {r.total_series:>8} {int(r.breaks):>8} {r.break_rate:>11.1f}% ${r.total_variance:>16,.0f} {int(r.escalated):>10}')

VARIANCE TREND BY MONTH
Period       Series   Breaks   Break Rate     Total Variance  Escalated
--------------------------------------------------------------------
2024-01          30        8        26.7% $          12,325          0
2024-02          30       10        33.3% $          29,348          1
2024-03          30        4        13.3% $          18,657          0
2024-04          30        8        26.7% $          11,827          0
2024-05          30        8        26.7% $          25,437          1
2024-06          30        9        30.0% $           2,732          1
2024-07          30        9        30.0% $          13,790          3
2024-08          30        6        20.0% $           5,687          1
2024-09          30        3        10.0% $           4,252          0
2024-10          30        7        23.3% $          14,115          0
2024-11          30        7        23.3% $          13,321          2
2024-12          30        8        26.7% $           

In [10]:
# Step 10 — Audit Schedule Export (Excel)

with pd.ExcelWriter('audit_schedules_2024.xlsx', engine='openpyxl') as writer:

    # Sheet 1: Revenue Summary
    pnl_export = pnl[['fund_id','fund_name','fund_type',
                       'total_gross_revenue','total_waivers','total_net_revenue',
                       'total_costs','operating_profit','profit_margin','avg_fee_bps']].copy()
    pnl_export.to_excel(writer, sheet_name='Revenue Summary', index=False)

    # Sheet 2: Monthly Accruals Detail
    monthly_export = monthly[['year_month_str','fund_id','fund_name','fund_type','share_class',
                               'avg_aum','month_end_aum','total_accrual','fee_waiver',
                               'net_revenue','avg_fee_bps','trading_days']].copy()
    monthly_export.columns = ['Period','Fund ID','Fund Name','Type','Share Class',
                               'Avg AUM','Month-End AUM','Gross Accrual','Fee Waiver',
                               'Net Revenue','Avg Fee (bps)','Trading Days']
    monthly_export.to_excel(writer, sheet_name='Monthly Accruals', index=False)

    # Sheet 3: Reconciliation Detail
    recon_df[['period','fund_id','share_class','blk_accrual','admin_accrual',
              'variance','variance_pct','is_break','break_cause','status']].to_excel(
        writer, sheet_name='Reconciliation', index=False)

    # Sheet 4: Breaks Only
    recon_df[recon_df['is_break']].sort_values('variance', key=abs, ascending=False)[
        ['period','fund_id','fund_name','share_class','blk_accrual','admin_accrual',
         'variance','variance_pct','break_cause','status','resolved_cause']
    ].to_excel(writer, sheet_name='Breaks & Exceptions', index=False)

    # Sheet 5: Journal Entries
    journal_df[['je_id','period','fund_id','share_class','account',
                'debit','credit','description']].to_excel(
        writer, sheet_name='Journal Entries', index=False)

    # Sheet 6: P&L Variance vs Admin
    pnl_variance = recon_df.groupby('fund_id').agg(
        blk_total   = ('blk_accrual', 'sum'),
        admin_total = ('admin_accrual', 'sum'),
    ).reset_index()
    pnl_variance['total_variance'] = pnl_variance['blk_total'] - pnl_variance['admin_total']
    pnl_variance['variance_bps']   = (pnl_variance['total_variance'] / pnl_variance['blk_total'] * 10000).round(2)
    pnl_variance.to_excel(writer, sheet_name='PnL Variance Analysis', index=False)

print('audit_schedules_2024.xlsx exported — 6 sheets')
print('  Sheet 1: Revenue Summary')
print('  Sheet 2: Monthly Accruals (1,080 rows)')
print('  Sheet 3: Reconciliation Detail (1,080 rows)')
print('  Sheet 4: Breaks & Exceptions')
print('  Sheet 5: Journal Entries')
print('  Sheet 6: P&L Variance Analysis')

audit_schedules_2024.xlsx exported — 6 sheets
  Sheet 1: Revenue Summary
  Sheet 2: Monthly Accruals (1,080 rows)
  Sheet 3: Reconciliation Detail (1,080 rows)
  Sheet 4: Breaks & Exceptions
  Sheet 5: Journal Entries
  Sheet 6: P&L Variance Analysis


In [11]:
# Step 11 — Export Dashboard Data & Final Summary

monthly_rev_total = monthly.groupby('year_month_str').agg(
    gross  = ('total_accrual','sum'),
    net    = ('net_revenue','sum'),
    waiver = ('fee_waiver','sum'),
).reset_index()

dashboard = {
    'summary': {
        'total_annual_gross':     round(float(monthly['total_accrual'].sum()), 0),
        'total_annual_waivers':   round(float(monthly['fee_waiver'].sum()), 0),
        'total_annual_net':       round(float(monthly['net_revenue'].sum()), 0),
        'total_operating_profit': round(float(pnl['operating_profit'].sum()), 0),
        'overall_margin_pct':     round(float(pnl['operating_profit'].sum()/pnl['total_net_revenue'].sum()*100), 1),
        'total_aum_bn':           round(float(aum_df[aum_df['date']==aum_df['date'].max()]['aum'].sum()/1e9), 2),
        'n_funds':                len(FUNDS),
        'n_share_classes':        len(SHARE_CLASSES),
        'monthly_calculations':   len(monthly),
        'total_recon_series':     total,
        'break_count':            breaks,
        'pre_recon_accuracy':     round(matched/total*100, 1),
        'post_recon_accuracy':    round((matched+resolved)/total*100, 1),
        'escalated':              escalated,
        'je_count':               len(journal_df),
    },
}

with open('dashboard_data.json', 'w') as f:
    json.dump(dashboard, f, indent=2)

recon_df.to_csv('reconciliation_full.csv', index=False)
monthly.to_csv('monthly_accruals.csv', index=False)

s = dashboard['summary']
print('PROJECT COMPLETE')
print('='*55)
print(f'  Funds:                     {s["n_funds"]} x {s["n_share_classes"]} share classes')
print(f'  Monthly calculations:      {s["monthly_calculations"]:,}')
print(f'  Annual gross accrual:      ${s["total_annual_gross"]/1e6:.2f}M')
print(f'  Annual net revenue:        ${s["total_annual_net"]/1e6:.2f}M')
print(f'  Operating profit:          ${s["total_operating_profit"]/1e6:.2f}M  ({s["overall_margin_pct"]}% margin)')
print(f'  Total AUM:                 ${s["total_aum_bn"]}B')
print(f'  Recon series:              {s["total_recon_series"]:,}')
print(f'  Breaks flagged:            {s["break_count"]} ({s["break_count"]/s["total_recon_series"]*100:.1f}%)')
print(f'  Pre-investigation accuracy:{s["pre_recon_accuracy"]}%')
print(f'  Post-resolution accuracy:  {s["post_recon_accuracy"]}%')
print(f'  Escalated:                 {s["escalated"]}')
print(f'  Journal entries:           {s["je_count"]:,}')

PROJECT COMPLETE
  Funds:                     10 x 3 share classes
  Monthly calculations:      360
  Annual gross accrual:      $105.62M
  Annual net revenue:        $104.73M
  Operating profit:          $92.31M  (88.1% margin)
  Total AUM:                 $45.36B
  Recon series:              360
  Breaks flagged:            87 (24.2%)
  Pre-investigation accuracy:75.8%
  Post-resolution accuracy:  97.5%
  Escalated:                 9
  Journal entries:           755
